In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
files = [
    "yellow_tripdata_2025-09.parquet",
    "yellow_tripdata_2025-10.parquet",
    "yellow_tripdata_2025-11.parquet"
]

dfs = [pd.read_parquet(f) for f in files]

df = pd.concat(dfs, ignore_index=True)

print(df.shape)
df.head()

(12861158, 20)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2025-09-01 00:19:20,2025-09-01 00:45:17,1.0,9.92,1.0,N,138,114,1,42.9,6.0,0.5,10.73,0.0,1.0,66.13,2.5,1.75,0.75
1,2,2025-09-01 00:15:20,2025-09-01 00:26:08,2.0,6.82,1.0,N,93,157,1,26.8,1.0,0.5,5.86,0.0,1.0,35.16,0.0,0.00,0.00
2,2,2025-09-01 00:06:07,2025-09-01 00:22:23,1.0,3.95,1.0,N,68,13,1,19.8,1.0,0.5,5.11,0.0,1.0,30.66,2.5,0.00,0.75
3,2,2025-09-01 00:49:47,2025-09-01 01:04:49,1.0,3.14,1.0,N,234,87,1,17.7,1.0,0.5,3.52,0.0,1.0,26.97,2.5,0.00,0.75
4,2,2025-09-01 00:05:00,2025-09-01 00:15:32,6.0,2.81,1.0,N,230,151,1,14.9,1.0,0.5,4.13,0.0,1.0,24.78,2.5,0.00,0.75


In [3]:
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

In [4]:
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_day"] = df["tpep_pickup_datetime"].dt.dayofweek
df["pickup_date"] = df["tpep_pickup_datetime"].dt.date

In [8]:
df = df[
    (df["trip_distance"] > 0) &
    (df["trip_distance"] < 100) &
    (df["total_amount"] > 0) &
    (df["total_amount"] < 500)
]

hourly_df = df.set_index("tpep_pickup_datetime").resample("H").agg({
    "trip_distance":"count",
    "total_amount":"mean"
})

hourly_df.rename(columns={
    "trip_distance":"trips",
    "total_amount":"avg_fare"
}, inplace=True)

hourly_df = hourly_df.dropna()

hourly_df = hourly_df.reset_index()

hourly_df.to_csv("hourly_taxi_demand.csv", index=False)